# Phase 3: Advanced Python for AI Systems — Comprehensive Beginner Guide

Welcome to Phase 3! In Phase 1 and Phase 2, you learned core Python syntax, data structures, built-in functions, OOP, and file handling. Now, you are stepping into **Advanced Python**—the exact engineering techniques required to build production-grade AI systems, Large Language Model (LLM) agents, asynchronous API pipelines, and robust machine learning tools.

Even if you are fresh to programming, every advanced topic below is broken down step-by-step from scratch so you understand **what** it is, **how** it works, and **why** AI developers use it.

### What You Will Learn in Phase 3:
1. **Decorators, Closures & `functools.wraps`** (Adding logging, retries, and timing to functions without changing their source code)
2. **Static Type Hinting & Pydantic Schema Validation** (`typing`, `TypedDict`, `Protocol`, and Pydantic for strict runtime LLM outputs)
3. **Concurrency & Parallelism** (`asyncio`, `threading`, and `multiprocessing` for fast, non-blocking AI applications)
4. **Subprocess Management** (`subprocess` and `sys` for executing terminal commands and checking GPU/system status)
5. **Memory Management** (`copy`: Understanding shallow vs. deep copies to prevent state corruption in neural network weights)
6. **Dynamic Code Execution & Sandboxing** (`eval`, `exec`, `compile` for building AI coding agents that write and test code)
7. **Packaging & Automated Testing** (Structuring Python projects and using `unittest` / `pytest` with `MagicMock` to test AI code without hitting costly APIs)

---
## Lesson 1: Decorators, Closures & `functools.wraps`

### What is a Decorator?
Imagine you have a function that calls an LLM API. You want to measure how many milliseconds it takes, and if the internet drops, you want it to automatically retry 3 times. Instead of copying and pasting retry code into every single function, you wrap your function in a **Decorator** (written using the `@` symbol above a function definition).

### How It Works:
A decorator is a function that takes *another* function as input, adds extra behavior around it, and returns the modified function. Using `functools.wraps` ensures that your decorated function doesn't lose its original name and documentation.

In [ ]:
import time
from functools import wraps

# A Parameterized Decorator for LLM API Retries & Latency Tracking
def retry_with_metrics(max_retries: int = 3):
    """Decorator factory that adds automatic retries and timing metrics to any function."""
    def decorator(func):
        @wraps(func)  # Preserves the original function's name and docstring
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_retries + 1):
                try:
                    start_time = time.perf_counter()
                    result = func(*args, **kwargs)  # Run the actual function
                    elapsed_ms = (time.perf_counter() - start_time) * 1000
                    print(f"[Success] {func.__name__} executed in {elapsed_ms:.2f} ms on attempt #{attempt}")
                    return result
                except Exception as e:
                    print(f"[Warning] {func.__name__} failed on attempt #{attempt}: {e}")
                    if attempt == max_retries:
                        print(f"[Error] All {max_retries} retries exhausted. Raising exception.")
                        raise
        return wrapper
    return decorator

# Applying our custom decorator to an LLM endpoint simulation function
@retry_with_metrics(max_retries=2)
def query_llm_api(prompt: str) -> dict:
    """Simulates sending a prompt to an AI model."""
    print(f"Sending prompt to model: '{prompt}'")
    return {"status": "ok", "response": f"Processed summary for: {prompt}"}

# Test running the decorated function
output = query_llm_api("Explain Transformer self-attention")
print("API Output Received:", output)

---
## Lesson 2: Static Type Hinting (`typing`) & Schema Validation (`Pydantic`)

### Why Type Hinting Matters in AI:
Python is dynamically typed, meaning variables can change types at any time. When building large AI systems (like multi-agent workflows), passing the wrong data type into a function causes confusing bugs. Type hints (`-> str`, `list[int]`) act as documentation and let code editors warn you before you even run your script.

### Key Tools:
1. **`typing` Module (`TypedDict`, `Protocol`):** Used to define structural shapes for dictionaries and interfaces.
2. **`Pydantic` (`BaseModel`, `Field`):** The industry standard for **runtime data validation**. When an LLM returns raw JSON text, Pydantic guarantees that the response matches your exact schema (e.g., ensuring a score is strictly between `0.0` and `1.0`).

In [ ]:
from typing import Optional, TypedDict, Protocol

# 1. TypedDict: Dict with strict expected keys and value types (used in LangGraph states)
class AgentWorkflowState(TypedDict):
    user_query: str
    retrieved_docs: list[str]
    step_count: int
    final_answer: Optional[str]

current_state: AgentWorkflowState = {
    "user_query": "What is RAG?",
    "retrieved_docs": ["Doc 1", "Doc 2"],
    "step_count": 1,
    "final_answer": None
}
print("TypedDict State Object:", current_state)

# 2. Protocol: Structural Subtyping (Duck Typing interface for Vector Databases)
class VectorDatabase(Protocol):
    def search(self, query: str, k: int) -> list[str]:
        ...

class MockQdrantDB:
    def search(self, query: str, k: int = 3) -> list[str]:
        return [f"Result {i} for '{query}'" for i in range(1, k + 1)]

def perform_rag_pipeline(db: VectorDatabase, query: str) -> list[str]:
    return db.search(query, k=2)

print("Vector Search Results:", perform_rag_pipeline(MockQdrantDB(), "fine-tuning"))

# 3. Pydantic Validation for LLM Structured Outputs
try:
    from pydantic import BaseModel, Field

    class SentimentAnalysis(BaseModel):
        sentiment: str = Field(description="positive, negative, or neutral")
        confidence: float = Field(ge=0.0, le=1.0, description="Confidence score between 0 and 1")

    # Validating data returned by an LLM
    llm_json_output = '{"sentiment": "positive", "confidence": 0.94}'
    validated_result = SentimentAnalysis.model_validate_json(llm_json_output)
    print("Pydantic Validated Object:", validated_result)
    print("Model Dump JSON:", validated_result.model_dump_json())
except ImportError:
    print("Pydantic is not installed in this environment. Run: pip install pydantic")

---
## Lesson 3: Concurrency & Parallelism (`asyncio`, `threading`, `multiprocessing`)

### The Big Problem: Waiting on APIs
When your AI app calls an LLM API (like OpenAI, Anthropic, or Gemini), your Python code has to wait 1 to 3 seconds for the network response. If you ask 10 questions in a row sequentially, it takes 30 seconds!

### How Concurrency Solves This:
* **`asyncio` (`async`/`await`):** The gold standard for **I/O-bound tasks** (waiting on network requests, databases, or LLM APIs). It lets your program send 50 API requests **at the exact same time** and handle responses as they arrive.
* **`threading`:** Uses multiple system threads for concurrent tasks.
* **`multiprocessing`:** Uses multiple CPU cores for heavy mathematical computation (like preprocessing massive datasets).

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

# 1. Asyncio: Non-blocking asynchronous LLM API calls
async def mock_async_llm_call(request_id: int) -> dict:
    print(f"[Start] Request #{request_id} sent to API...")
    await asyncio.sleep(0.1)  # Simulates waiting for network response without blocking CPU
    print(f"[Done]  Request #{request_id} received!")
    return {"id": request_id, "status": "success"}

async def run_concurrent_pipeline():
    # asyncio.gather fires off all 4 API requests simultaneously!
    tasks = [mock_async_llm_call(i) for i in range(1, 5)]
    start = time.perf_counter()
    results = await asyncio.gather(*tasks)
    elapsed = (time.perf_counter() - start) * 1000
    print(f"All 4 requests completed concurrently in {elapsed:.2f} ms total!")
    return results

# In Jupyter notebooks, top-level await is supported natively
async_output = await run_concurrent_pipeline()
print("Async Results:", async_output)

# 2. ThreadPoolExecutor: Running synchronous blocking tasks concurrently
def download_file_chunk(chunk_id: int) -> str:
    time.sleep(0.05)  # Simulate file reading delay
    return f"chunk_{chunk_id}_data"

with ThreadPoolExecutor(max_workers=3) as executor:
    chunks = list(executor.map(download_file_chunk, range(3)))
print(
    "Threadpool Chunks Downloaded:", chunks)

---
## Lesson 4: Subprocess Management (`subprocess`, `sys`)

### Why Use `subprocess`?
Sometimes your Python script needs to talk directly to your operating system terminal. For example:
* Checking if an NVIDIA GPU is available (`nvidia-smi`)
* Extracting audio from a video file using `ffmpeg` before feeding it to Whisper
* Running shell commands inside your Python automation scripts

`subprocess.run()` lets you execute terminal commands safely from inside Python and capture their output text.

In [ ]:
import subprocess
import sys

# Executing a terminal command safely from Python (checking the active Python version)
try:
    completed_process = subprocess.run(
        [sys.executable, "--version"],
        capture_output=True,  # Captures printed terminal output
        text=True,            # Returns output as a string instead of raw bytes
        check=True            # Raises an error if the command fails
    )
    print("Subprocess successfully executed! Output:", completed_process.stdout.strip())
except Exception as e:
    print("Subprocess execution failed:", e)

---
## Lesson 5: Memory Management (`copy`: Shallow vs. Deep Copy)

### The Python Assignment Trap (`=`):
In Python, when you assign a variable to another (`dict2 = dict1`), **you are not copying the data—you are just giving the same data a second name**. If you modify `dict2`, `dict1` changes too!

### Shallow vs. Deep Copy:
* **Shallow Copy (`copy.copy()`):** Copies the outer container, but nested dictionaries or lists still point to the original memory locations.
* **Deep Copy (`copy.deepcopy()`):** Recursively clones **everything**, creating a 100% independent duplicate in RAM.
* **Why it matters in AI:** When saving model state checkpoints or tracking multi-agent conversation history across steps, failing to use `deepcopy` will result in corrupted training states!

In [ ]:
import copy

# Original AI training state dictionary containing nested weights
training_state = {
    "epoch": 1,
    "optimizer_weights": [0.15, 0.88, 0.42]
}

# 1. Shallow copy vs Deep copy
shallow_clone = copy.copy(training_state)
deep_clone = copy.deepcopy(training_state)

# Mutating the nested list in the original state
training_state["optimizer_weights"][0] = 999.0

print("Original State:     ", training_state["optimizer_weights"])
print("Shallow Clone (Leaked):", shallow_clone["optimizer_weights"])  # Affected by mutation!
print("Deep Clone (Safe):    ", deep_clone["optimizer_weights"])      # 100% independent!

---
## Lesson 6: Dynamic Code Execution & Sandboxing (`eval`, `exec`, `compile`)

### Building AI Coding Agents
Advanced AI systems (like OpenAI's Advanced Data Analysis or custom coding agents) work by **writing Python code on the fly and executing it**. Python provides built-in tools for this:
* **`eval(expr)`:** Evaluates a single mathematical expression string (e.g., `eval("2 + 2")`).
* **`exec(code_string)`:** Executes blocks of multi-line Python code.
* **`compile()`:** Compiles code into bytecode before execution.

⚠️ **Security Warning:** Never run raw `exec()` or `eval()` on untrusted text without restricting built-in functions, otherwise malicious code could wipe your hard drive!

In [ ]:
import math

# 1. Restricted eval() for safe mathematical evaluation
# Passing an empty dictionary for '__builtins__' blocks dangerous commands like open() or import os
safe_globals = {"__builtins__": {}, "sqrt": math.sqrt}
expression = "sqrt(169) * 2"
result = eval(expression, safe_globals, {})
print(f"Evaluated expression '{expression}' safely -> Result: {result}")

# 2. compile() + exec() for running LLM-generated code snippets
llm_generated_python_code = """
def compute_f1_score(precision: float, recall: float) -> float:
    return 2 * (precision * recall) / (precision + recall)

computed_result = round(compute_f1_score(0.90, 0.80), 4)
"""

# Compile code into bytecode
bytecode = compile(llm_generated_python_code, filename="<llm_sandbox>", mode="exec")

# Execute inside a sandboxed local namespace with restricted built-ins
sandbox_globals = {"__builtins__": {"round": round}}
sandbox_locals = {}

exec(bytecode, sandbox_globals, sandbox_locals)
print("Executed LLM Code Output (F1 Score):", sandbox_locals["computed_result"])

---
## Lesson 7: Packaging & Automated Testing (`unittest`, `MagicMock`)

### Why Test AI Code?
When writing complex AI agents and data pipelines, code breaks silently. Professional developers write automated unit tests using Python's built-in **`unittest`** framework.

### What is Mocking (`unittest.mock`)?
If you want to test a function that calls an LLM API, you don't want to make 50 real API calls every time you test your code (which costs money and time). **`MagicMock`** lets you fake the API response so you can test your logic instantly and safely offline!

In [ ]:
import unittest
from unittest.mock import MagicMock

# A class representing an expensive LLM client
class LLMClient:
    def generate_response(self, prompt: str) -> str:
        # Real API call that costs money/time
        raise RuntimeError("Network error: Should not hit real API during unit tests!")

def summarize_document(client: LLMClient, text: str) -> str:
    """Function we want to test."""
    if not text.strip():
        raise ValueError("Document text cannot be empty")
    return client.generate_response(f"Summarize this: {text}")

# Writing our Unit Test Suite
class TestAIPipeline(unittest.TestCase):
    def test_summarize_with_mock_client(self):
        # Create a mock client instead of hitting the real API
        mock_client = LLMClient()
        mock_client.generate_response = MagicMock(return_value="Mocked Summary Result")

        # Run our function using the mock client
        result = summarize_document(mock_client, "Artificial Intelligence is evolving rapidly.")

        # Assertions: Verify our code works as expected
        self.assertEqual(result, "Mocked Summary Result")
        mock_client.generate_response.assert_called_once_with("Summarize this: Artificial Intelligence is evolving rapidly.")
        print("Unit test passed successfully!")

    def test_empty_input_raises_error(self):
        # Verify that passing empty text correctly raises a ValueError
        with self.assertRaises(ValueError):
            summarize_document(LLMClient(), "   ")

# Run the unit test suite directly inside the Jupyter Notebook
suite = unittest.TestLoader().loadTestsFromTestCase(TestAIPipeline)
unittest.TextTestRunner(verbosity=2).run(suite)